# Notebook 06 of 7 — Backtest + Validation (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1442](https://github.com/prajoria/OpenBB/issues/1442) · Track A counterpart: [`../portfolio/06-backtest-and-validation.ipynb`](../portfolio/06-backtest-and-validation.ipynb).

---

## Where we are in Sam's story

NB05 (Track B) paper-traded three intuitive names on top of the free signal shape from NB04. Before I graduate any of it to a real order, I want to know whether the *strategy* — 'hold this basket, rebalance to top-K momentum monthly' — has ever actually worked, or whether NB05 was one lucky what-if. Same question Track A NB06 asked. Same engine. The only knob I turn is the provider: CBOE historicals in place of `fmp_cached`.

By the end we can answer:

> *Does the strategy have real edge under free-only historicals — or does the answer flip once we drop paid total-return-adjusted prices?*


## 0. Before we run anything

Same venv rule as every notebook in the series — `.venv_portfolio`. State goes into `.notebook_state/` (gitignored). We pickle to `backtest_result_free.pkl` and export the tearsheet to `tearsheet_free.html` — do NOT overwrite Track A's `backtest_result.pkl` / `tearsheet.html`.


In [1]:
# [Track B / NB06 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only NB06 looks different from Track A

The backtest engine itself — `openbb_backtest.run / sweep / validate / tearsheet / factor_eval / bundle` — is part of the free extension. Every call in this notebook is a straight copy of the Track A shape. What changes is a single kwarg: `provider="cboe"` threaded through every engine call so the historical price source is free-authoritative rather than `fmp_cached`.

**Provider chain for this notebook:**

| Data path | Provider (Track B) |
|---|---|
| Equity historical (rebase & fills) | `cboe` `EquityHistorical` (EOD) |
| Benchmark historical (SPY) | `cboe` |
| Sweep / validate / factor_eval inputs | `cboe` |
| Tearsheet inputs | `cboe` |
| Bundle ingest | `cboe` |

**The adjusted-close gotcha that matters.** FMP reports total-return-adjusted close (dividends re-invested). CBOE reports price-only. Backtest CAGR / Sharpe / MaxDD numbers will differ by roughly 1-2 percentage points per year between the two providers, and the delta is largest on dividend-paying names (BND, VNQ, GLD, VTI). That's not the engine — that's the data. Any headline number in this notebook is labelled as CBOE-sourced so you can reconcile against Track A directly. §9 pickles both the numbers and the provider tag.

**Sweep caveat.** The `momentum_12_1` strategy defaults look back 252 trading days. If the CBOE historical range for the chosen window is shorter than lookback + a warm-up buffer, the sweep will trim to what's available or emit an empty grid; the notebook prints the shape so the reader sees what happened rather than a silent zero (Testing Rule #3).

Bare-term pointers (all cited with Investopedia links in Track A NB06; not re-cited here): backtesting, look-ahead bias, survivorship bias, sample selection bias, data mining bias, equity curve, CAGR, portfolio turnover, rebalancing, overfitting, tear sheet, factor investing, momentum, quintile, alpha, Jensen's alpha, information ratio, Fama-French three-factor.


## 1. Load basket + build the config

Basket comes from NB01 (shared with Track A). If `basket.json` is missing, fall back to the STORY_BIBLE-locked shape so the notebook runs standalone. Then build the same `BacktestConfig` shape as Track A NB06 §1 — universe, window, benchmark, commissions=0, slippage=0.


In [2]:
# [Track B / NB06 §1] Load basket + build BacktestConfig
import json
from datetime import date
from decimal import Decimal
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)

BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12},
    {"symbol": "NVDA",  "weight": 0.10},
    {"symbol": "GOOGL", "weight": 0.08},
    {"symbol": "AAPL",  "weight": 0.08},
    {"symbol": "AMD",   "weight": 0.06},
    {"symbol": "QQQ",   "weight": 0.15},
    {"symbol": "VTI",   "weight": 0.20},
    {"symbol": "VNQ",   "weight": 0.08},
    {"symbol": "BND",   "weight": 0.10},
    {"symbol": "GLD",   "weight": 0.03},
]
bp = STATE / "basket.json"
if bp.exists():
    basket = json.loads(bp.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {bp} ({len(basket)} names, shared with Track A)")
else:
    basket = BASKET_LOCKED
    print("basket.json missing — regenerated from STORY_BIBLE locked list")

UNIVERSE = [p["symbol"] for p in basket]
START, END = date(2023, 1, 3), date(2024, 12, 31)
PROVIDER = "cboe"   # free-authoritative EOD; see §0.5 for the adjusted-close caveat

config = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE,
    start=START,
    end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS = {"symbols": UNIVERSE}
print(f"Strategy:     {config.strategy}")
print(f"Universe:     {len(config.universe)} symbols")
print(f"Window:       {config.start} → {config.end}")
print(f"Benchmark:    {config.benchmark}")
print(f"Provider:     {PROVIDER}  (free-authoritative EOD, price-only)")


Loaded basket from .notebook_state\basket.json (10 names, shared with Track A)
Strategy:     buy_and_hold
Universe:     10 symbols
Window:       2023-01-03 → 2024-12-31
Benchmark:    SPY
Provider:     cboe  (free-authoritative EOD, price-only)


## 2. Run — `obb.backtest.run(..., provider='cboe')`

Same call shape as Track A NB06 §2, only difference is `provider=` — the backtest engine passes the kwarg through to whichever equity historical fetcher it uses to build the fills tape. Free-only run produces the same shape of output — equity curve, per-day cash / exposure, and a summary metrics object (Sharpe, MaxDD, CAGR, volatility, Sortino, Calmar).

Expect the numbers to differ from Track A by ~1-2%/yr on Sharpe/CAGR because CBOE is price-only (no dividend reinvestment).


In [3]:
# [Track B / NB06 §2] obb.backtest.run(..., provider='cboe') — free-only historicals
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from openbb import obb

result_obj = obb.backtest.run(config, strategy_params=STRATEGY_PARAMS, provider=PROVIDER)
result = result_obj.results

m = result.metrics
print(f"Summary metrics (provider={PROVIDER}):")
for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
    val = getattr(m, field, None)
    if val is None:
        continue
    print(f"  {field:<22} {float(val):+.4f}")

eq = result.equity_curve
if hasattr(eq, "to_df"):
    eq_df = eq.to_df()
elif isinstance(eq, list):
    eq_df = pd.DataFrame([row.model_dump() if hasattr(row,"model_dump") else row for row in eq])
else:
    eq_df = pd.DataFrame(eq)

print(f"\nEquity curve — {len(eq_df)} rows (engine={result.engine_used})")
print("Head:")
print(eq_df.head(3).to_string())
print("Tail:")
print(eq_df.tail(3).to_string())

# Loud-empty guard (Testing Rule #3): non-empty universe but empty curve = shape bug.
if len(UNIVERSE) > 0 and len(eq_df) == 0:
    raise RuntimeError(
        f"backtest.run returned an empty equity curve for {len(UNIVERSE)} symbols "
        f"under provider={PROVIDER}; CBOE historical shape may not match backtest "
        f"engine expectations."
    )


Summary metrics (provider=cboe):
  sharpe                 +2.1470
  volatility             +0.1671
  max_drawdown           -0.1142
  cagr                   +0.4115
  sortino                +3.4381
  calmar                 +3.6043

Equity curve — 502 rows (engine=vectorized)
Head:
                       date     equity       cash  exposure
0 2023-01-03 00:00:00+00:00  100000.00  100000.00       0.0
1 2023-01-04 00:00:00+00:00  100463.05       0.00       1.0
2 2023-01-05 00:00:00+00:00   98474.19       0.00       1.0
Tail:
                         date     equity  cash  exposure
499 2024-12-27 00:00:00+00:00  201532.71  0.00       1.0
500 2024-12-30 00:00:00+00:00  199891.26  0.00       1.0
501 2024-12-31 00:00:00+00:00  198682.48  0.00       1.0


## 3. Sweep — momentum grid under CBOE

Same 2×2 grid as Track A NB06 §3 (`lookback` in {126, 252} × `gross` in {0.75, 1.0}) on the `momentum_12_1` strategy. buy_and_hold has no meaningful params to sweep. The sweep goal is unchanged: is the best-cell Sharpe a peak on a wide plateau, or a lonely spike?


In [4]:
# [Track B / NB06 §3] obb.backtest.sweep — momentum grid under provider='cboe'
import warnings; warnings.filterwarnings("ignore")
from copy import deepcopy

sweep_cfg = deepcopy(config)
sweep_cfg.strategy = "momentum_12_1"
param_grid = {
    "symbols":  [UNIVERSE],
    "lookback": [126, 252],
    "gross":    [0.75, 1.0],
}
sweep_obj = obb.backtest.sweep(sweep_cfg, param_grid=param_grid, rank_by="sharpe", provider=PROVIDER)
sweep = sweep_obj.results
rows = sweep.results

if len(rows) == 0:
    print(f"Sweep returned 0 rows under provider={PROVIDER}.")
    print("  Likely cause: CBOE window too short for lookback=252 warm-up.")
    print("  Fallback: continuing with base-run metrics only.")
    best_params = {}
    best_sharpe = float("nan")
else:
    print(f"Sweep: {len(rows)} runs, ranked by {sweep.rank_by} (best first)")
    print()
    print(f"{'lookback':>10}{'gross':>8}{'sharpe':>10}{'max_dd':>10}{'cagr':>10}")
    print("-" * 48)
    ranked = sorted(rows, key=lambda r: float(r.metrics.sharpe), reverse=True)
    for row in ranked:
        p, mx = row.params, row.metrics
        print(f"{p.get('lookback','?'):>10}{p.get('gross','?'):>8}"
              f"{float(mx.sharpe):>10.3f}{float(mx.max_drawdown):>10.3f}"
              f"{float(mx.cagr):>10.3f}")
    best_params = {k: v for k, v in sweep.best.items() if k != "symbols"}
    best_sharpe = float(sweep.best_metrics.sharpe)
    print(f"\nBest params: {best_params}")
    print(f"Best sharpe: {best_sharpe:+.3f}")


Sweep: 4 runs, ranked by sharpe (best first)

  lookback   gross    sharpe    max_dd      cagr
------------------------------------------------
       252     1.0     1.792    -0.128     0.430
       252    0.75     1.792    -0.097     0.313
       126    0.75     1.597    -0.091     0.245
       126     1.0     1.597    -0.120     0.333

Best params: {'lookback': 252, 'gross': 1.0}
Best sharpe: +1.792


## 4. Validate — walk-forward + PBO under CBOE

Same `wfo` call as Track A NB06 §4. Walk-forward retrains on rolling in-sample windows and reports out-of-sample Sharpe per fold; PBO — Probability of Backtest Overfitting — collapses the whole family into one scalar. The PBO reading rule is unchanged from Track A: < 0.30 reasonably robust, 0.30-0.50 borderline, > 0.50 coin flip.


In [5]:
# [Track B / NB06 §4] obb.backtest.validate — walk-forward under provider='cboe'
import warnings; warnings.filterwarnings("ignore")

val_obj = obb.backtest.validate(
    config, method="wfo", strategy_params=STRATEGY_PARAMS, provider=PROVIDER,
)
val = val_obj.results

print(f"Walk-forward validation ({val.method}, provider={PROVIDER}):")
print(f"  n_folds:   {len(val.folds)}")
pbo = getattr(val, "pbo", None)
if pbo is not None:
    print(f"  PBO:       {float(pbo):.3f}   (< 0.30 = reasonably robust)")
dsr = getattr(val, "deflated_sharpe", None)
if dsr is not None:
    print(f"  DSR:       {float(dsr):+.3f}")
if hasattr(val, "verdict"):
    print(f"  verdict:   {val.verdict}")

print("\nPer-fold Sharpe:")
for i, fold in enumerate(val.folds):
    fm = fold.metrics if hasattr(fold, "metrics") else fold
    s = float(getattr(fm, "sharpe", float("nan")))
    print(f"  fold {i:>2}: sharpe={s:+.3f}")


Walk-forward validation (wfo, provider=cboe):
  n_folds:   2
  PBO:       0.270   (< 0.30 = reasonably robust)
  DSR:       +0.000
  verdict:   overfit

Per-fold Sharpe:
  fold  0: sharpe=+3.273
  fold  1: sharpe=+0.380


## 5. Tearsheet — HTML export under CBOE

Same `obb.backtest.tearsheet(export=True)` call as Track A NB06 §6. QuantStats-style HTML: rolling Sharpe, monthly returns heatmap, drawdown periods, exposure. Written to `.notebook_state/tearsheet_free.html` — Track A's `tearsheet.html` is NOT touched, so the reader can diff the two side-by-side.


In [6]:
# [Track B / NB06 §5] obb.backtest.tearsheet — HTML export to *_free path
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

try:
    ts_obj = obb.backtest.tearsheet(
        config, export=True, strategy_params=STRATEGY_PARAMS, provider=PROVIDER,
    )
    ts = ts_obj.results
    src_path = (getattr(ts, "html_path", None)
                or getattr(ts, "artifact_path", None)
                or getattr(ts, "path", None))
    dest = Path(".notebook_state") / "tearsheet_free.html"
    if src_path and Path(src_path).exists():
        dest.write_bytes(Path(src_path).read_bytes())
        print(f"Tearsheet (repo-rel): {dest}  ({dest.stat().st_size:,} bytes)")
    else:
        html = getattr(ts, "html", None)
        if html:
            dest.write_text(html, encoding="utf-8")
            print(f"Tearsheet (repo-rel): {dest}  ({dest.stat().st_size:,} bytes) [inline]")
        else:
            print("WARNING: tearsheet did not produce an HTML artifact")
    print(f"\nRolling-Sharpe window: {getattr(ts, 'rolling_window', 21)} sessions")
    print("Track A's tearsheet.html NOT modified by this notebook.")
except (ImportError, ModuleNotFoundError) as exc:
    # Narrow guard per Testing Rule #3 — quantstats/matplotlib are optional deps.
    print(f"WARNING: tearsheet optional-dep missing — {type(exc).__name__}: {str(exc)[:120]}")
    print("  Fallback: continuing without HTML tearsheet artifact.")


Tearsheet (repo-rel): .notebook_state\tearsheet_free.html  (501,911 bytes)

Rolling-Sharpe window: 21 sessions
Track A's tearsheet.html NOT modified by this notebook.


## 6. Factor eval — Momentum quintiles under CBOE

Same `obb.backtest.factor_eval(factor='Momentum', quantiles=5)` call as Track A NB06 §7. Rank the universe by trailing momentum each period, compare top vs bottom quintile forward returns. If Q5 - Q1 is positive and monotone across quintiles, the factor discriminates on this universe under CBOE historicals.


In [7]:
# [Track B / NB06 §6] obb.backtest.factor_eval — Momentum quantile stats under CBOE
import warnings; warnings.filterwarnings("ignore")

try:
    fe_obj = obb.backtest.factor_eval(
        config, factor="Momentum", quantiles=5, periods=[1, 5, 21],
        provider=PROVIDER,
    )
    fe = fe_obj.results
    print(f"Factor: Momentum   quantiles=5   periods=[1,5,21]   provider={PROVIDER}")
    ic = getattr(fe, "ic", None) or getattr(fe, "information_coefficient", None)
    if ic is not None:
        try:
            print(f"  mean IC:     {float(ic):+.4f}")
        except (TypeError, ValueError):
            print(f"  IC (raw):    {ic}")
    qr = getattr(fe, "quantile_returns", None)
    if qr is not None:
        print("\n  Per-quantile mean forward returns:")
        try:
            for q, v in dict(qr).items():
                print(f"    q{q}: {float(v):+.5f}")
        except (TypeError, ValueError) as exc:
            print(f"    (raw shape) {type(qr).__name__} — coerce: {exc}")
    else:
        fields = list(fe.model_fields.keys()) if hasattr(fe, "model_fields") else "n/a"
        print(f"\n  fields available: {fields}")
except (ImportError, ModuleNotFoundError) as exc:
    print(f"WARNING: factor_eval optional-dep missing — {type(exc).__name__}: {exc}")
    print("  Fallback: skipping factor evaluation for this notebook run.")


Dropped 8.2% entries from factor data: 8.2% in forward returns computation and 0.0% in binning phase (set max_loss=0 to see potentially suppressed Exceptions).
max_loss is 35.0%, not exceeded: OK!


Factor: Momentum   quantiles=5   periods=[1,5,21]   provider=cboe

  Per-quantile mean forward returns:
    q1: -0.00117
    q2: -0.01239
    q3: -0.00450
    q4: +0.00203
    q5: +0.01604


## 7. Bundle — reproducibility freeze under CBOE

Same `obb.backtest.bundle.ingest` call as Track A NB06 §8. Freezes the exact CBOE-sourced inputs so the same run can be replayed months later on the same data — the reproducibility lifeline that makes the strategy debuggable in the future.


In [8]:
# [Track B / NB06 §7] obb.backtest.bundle.ingest — freeze CBOE-sourced inputs
import warnings; warnings.filterwarnings("ignore")

try:
    bundle_obj = obb.backtest.bundle.ingest(config, provider=PROVIDER)
    b = bundle_obj.results
    path = (getattr(b, "path", None)
            or getattr(b, "bundle_path", None))
    digest = (getattr(b, "hash", None)
              or getattr(b, "digest", None)
              or getattr(b, "content_hash", None))
    print("Bundle ingested (CBOE-sourced):")
    if path:
        from pathlib import Path
        p = Path(str(path))
        print(f"  path (name): {p.name}")
    if digest:
        print(f"  hash:        {str(digest)[:16]}…")
    if not path and not digest:
        fields = list(b.model_fields.keys()) if hasattr(b, "model_fields") else dir(b)[:10]
        print(f"  fields: {fields}")
except (ImportError, ModuleNotFoundError) as exc:
    print(f"WARNING: bundle.ingest optional-dep missing — {type(exc).__name__}: {exc}")
    print("  Fallback: continuing without frozen bundle artifact.")


Bundle ingested (CBOE-sourced):
  fields: ['name', 'symbols', 'calendar', 'start', 'end', 'ingested_at', 'has_fundamentals']


## 8. Save state for NB07 (Track B)

Pickle the summary result to `.notebook_state/backtest_result_free.pkl`. **Do NOT overwrite Track A's `backtest_result.pkl`** — NB07 Track B will read the `_free` variant, and the diff between the two is the story.


In [9]:
# [Track B / NB06 §8] Pickle backtest result — write ONLY to *_free.pkl
import pickle  # noqa: S403 — trusted local artifact under .notebook_state/
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
m = result.metrics

artifact = {
    "track": "B (free-only)",
    "provider": PROVIDER,
    "config": {
        "strategy":  config.strategy,
        "universe":  list(config.universe),
        "start":     str(config.start),
        "end":       str(config.end),
        "benchmark": config.benchmark,
    },
    "metrics": {
        f: float(getattr(m, f))
        for f in ("sharpe", "volatility", "max_drawdown",
                  "cagr", "sortino", "calmar")
        if getattr(m, f, None) is not None
    },
    "engine_used": result.engine_used,
    "equity_curve_rows": len(eq_df),
    "sweep_best": {
        "params": best_params,
        "sharpe": best_sharpe,
    },
    "validation": {
        "method":  val.method,
        "n_folds": len(val.folds),
        "pbo":     float(pbo) if pbo is not None else None,
    },
    "gap_vs_track_a": (
        "CBOE historicals are price-only (no dividend reinvestment); "
        "Track A's paid path uses total-return-adjusted close. "
        "Expect Sharpe/CAGR delta of ~1-2%/yr, largest on dividend-payers."
    ),
}

out = state / "backtest_result_free.pkl"
out.write_bytes(pickle.dumps(artifact))
print(f"Wrote (repo-rel): {out}  ({out.stat().st_size:,} bytes)")
print("Track A's backtest_result.pkl NOT modified by this notebook.")


Wrote (repo-rel): .notebook_state\backtest_result_free.pkl  (745 bytes)
Track A's backtest_result.pkl NOT modified by this notebook.


---

## What is NOT in this notebook

Same gaps as Track A NB06 plus the free-tier delta:

- **Monte-Carlo bootstrap.** Full bootstrap CI on backtest metrics is future work in both tracks.
- **Regime-conditional slicing.** `openbb_regime` wiring into `validate` ("PBO under bear vs bull") not shipped in either track.
- **Multi-strategy portfolios.** `run` handles one strategy; combining N with a risk budget is future work.
- **Total-return-adjusted historicals.** CBOE is price-only. Any   strategy whose alpha comes from dividend reinvestment (BND, VNQ, GLD,   income-tilted VTI) will look worse under Track B than Track A by   roughly 1-2 percentage points per year of CAGR. That's a data-source   fact, not an engine bug — if you want the total-return read, you need   the paid Track A path.
- **Intraday fills.** CBOE historicals are EOD; the fills tape is   daily bars. Intraday backtest granularity is a Track A concern.

## Preview of NB07 (Track B)

NB07 will replay the Track B pipeline offline against checked-in snapshots — same reproducibility discipline as Track A NB07, but sourced from the CBOE / yfinance recorded fixtures instead of `fmp_cached` cache. The diff between the two tracks' final HTML report is the story of what free-only actually costs.

## 📚 Further reading

Every Investopedia link cited in Track A NB06 (backtesting, look-ahead bias, survivorship, sample selection, data mining, equity curve, CAGR, portfolio turnover, rebalancing, overfitting, tear sheet, factor investing, momentum, quintile, alpha, Jensen's alpha, information ratio, Fama-French) applies unchanged. Not re-cited here.

**Canonical references** (unchanged from Track A):

- Bailey, Borwein, Lopez de Prado & Zhu — "The Probability of Backtest   Overfitting," 2014. SSRN:   <https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2326253>. The   PBO definition and the CSCV procedure `obb.backtest.validate`   implements.
- Bailey & Lopez de Prado — "The Deflated Sharpe Ratio," *JPM* 40(5),   2014. SSRN: <https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551>.   How to adjust a reported Sharpe for the number of variants tried.

**Free-authoritative sources used:**

- **CBOE `EquityHistorical`** — EOD price-only bars, threaded through   `openbb_backtest` via `provider='cboe'` on every engine call in this   notebook (§2, §3, §4, §5, §6, §7).
- **`openbb_backtest`** — engine (free extension, unchanged between   tracks).
